In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install umap-learn seaborn scikit-learn scipy

In [3]:
print("=" * 60)
print("PHASE 1: DOWNLOADING & PARSING DR-MUSIC DATA (UP TO p90)")
print("=" * 60)

import os
import re
import glob
import numpy as np
import pandas as pd
import warnings
from scipy.signal import detrend, butter, filtfilt
from scipy.fft import fft, fftfreq
from scipy.stats import skew, kurtosis
from huggingface_hub import snapshot_download

repo_path = snapshot_download(
    repo_id="BreathSense/BreathSense",
    repo_type="dataset",
    allow_patterns="data/**/*drmusic.csv"
)

all_files = glob.glob(f"{repo_path}/data/**/*drmusic.csv", recursive=True)
data_dict = {}

for file_path in all_files:
    activity = os.path.basename(os.path.dirname(file_path))
    match = re.search(r'(p\d+)', os.path.basename(file_path), re.IGNORECASE)
    if not match:
        continue
        
    person_id = match.group(1).lower()
    
    # --- NEW FILTER APPLIED HERE ---
    # Extract the integer from the ID and skip if it's strictly greater than 90
    person_num = int(person_id.replace('p', ''))
    if person_num > 90:
        continue
    # -------------------------------
        
    if person_id not in data_dict:
        data_dict[person_id] = {}
    data_dict[person_id][activity] = pd.read_csv(file_path).values.tolist()

final_df = pd.DataFrame.from_dict(data_dict, orient='index')
desired_columns = ['rest', 'walk', 'run', 'stairs']
existing_columns = [col for col in desired_columns if col in final_df.columns]
final_df = final_df[existing_columns]
final_df = final_df.loc[
    sorted(final_df.index, key=lambda x: int(x.replace('p', '')))
]

# ── Labels ────────────────────────────────────
# Ensure we only keep indices that actually exist in our truncated dataframe
gym_idx    = [x-1 for x in [2,5,7,10,13,20,30,33,40,44,48,50,52,63,64,68,69,79,82,86,89] if x <= 90]
sports_idx = [x-1 for x in [3,6,9,11,12,22,25,37,43,49,53,54,55,65,70,76,78,81,87,88] if x <= 90]
smoker_idx = [x-1 for x in [14,15,16,17,18,19,23,27,29,32,36,42,46,60,61,62,74,75] if x <= 90]

all_participants = set(range(len(final_df)))
normal_idx = list(all_participants - set(gym_idx) - set(sports_idx) - set(smoker_idx))

class_map = {}
for pid in normal_idx:  class_map[pid] = 0   # Normal
for pid in gym_idx:     class_map[pid] = 1   # Gym
for pid in sports_idx:  class_map[pid] = 2   # Sports
for pid in smoker_idx:  class_map[pid] = 3
CLASS_NAMES = {0: 'Normal', 1: 'Gym', 2: 'Sports', 3: 'Smoker'}

print(f"Loaded {len(final_df)} participants (Capped at p90)")
print(f"Normal: {len(normal_idx)}  Gym: {len(gym_idx)}  Sports: {len(sports_idx)}  Smoker: {len(smoker_idx)}")
print(f"Activities available: {existing_columns}")

PHASE 1: DOWNLOADING & PARSING DR-MUSIC DATA (UP TO p90)


Fetching ... files: 0it [00:00, ?it/s]

Loaded 90 participants (Capped at p90)
Normal: 31  Gym: 21  Sports: 20  Smoker: 18
Activities available: ['rest', 'walk', 'run', 'stairs']


In [4]:
print("=" * 60)
print("PHASE 2: TRIMMING INITIAL SENSOR ARTIFACTS")
print("=" * 60)

# 2 seconds = 20 rows (assuming a 10 Hz sampling rate)
ROWS_TO_REMOVE = 20

def trim_sensor_mistakes(sensor_data):
    # Check if the cell contains actual data (a list) and not NaN
    if isinstance(sensor_data, list):
        # Remove the first 20 rows if the recording is long enough
        if len(sensor_data) > ROWS_TO_REMOVE:
            return sensor_data[ROWS_TO_REMOVE:]
        else:
            # If the recording is corrupted/too short, return an empty list
            return [] 
    return sensor_data

# Apply the trimming function to every column (activity) in the dataframe
for activity in final_df.columns:
    final_df[activity] = final_df[activity].apply(trim_sensor_mistakes)

print(f"Successfully removed the first {ROWS_TO_REMOVE} rows (2 seconds) from all activities.")

# Optional verification: Check the length of the first person's data to confirm
first_person = final_df.index[0]
first_activity = final_df.columns[0]
sample_data = final_df.loc[first_person, first_activity]

if isinstance(sample_data, list):
    print(f"Verification: {first_person} '{first_activity}' data now has {len(sample_data)} rows remaining.")

PHASE 2: TRIMMING INITIAL SENSOR ARTIFACTS
Successfully removed the first 20 rows (2 seconds) from all activities.
Verification: p1 'rest' data now has 280 rows remaining.


In [5]:
print("=" * 60)
print("PHASE 3: RESPIRATORY FEATURE EXTRACTION")
print("=" * 60)

import numpy as np
from scipy.signal import find_peaks, welch
from scipy.stats import skew, entropy

def extract_features(signal, fs=10.0):
    """
    Extracts 11 respiratory features from a 1D signal array.
    Assumes a sampling frequency (fs) of 10 Hz based on prior data trimming.
    """
    # Convert to numpy array and ensure it's 1D
    sig = np.array(signal, dtype=float).flatten()
    
    # Check if the signal is long enough to extract meaningful features (e.g., > 5 seconds)
    if len(sig) < fs * 5:
        return np.full(11, np.nan) # Return NaNs if data is missing or too short

    # --- Time Domain Preparations ---
    # Find peaks (end of inhalation) and troughs (end of exhalation)
    # Using a distance of fs * 1.5 assumes a maximum breathing rate of ~40 breaths/min
    peaks, _ = find_peaks(sig, distance=int(fs * 1.5))
    troughs, _ = find_peaks(-sig, distance=int(fs * 1.5))
    
    # Time vector for slope calculations
    t = np.arange(len(sig)) / fs

    # --- 1. Peak to Peak Amplitude ---
    if len(peaks) > 0 and len(troughs) > 0:
        mean_peak = np.mean(sig[peaks])
        mean_trough = np.mean(sig[troughs])
        ptp_amplitude = mean_peak - mean_trough
    else:
        ptp_amplitude = np.max(sig) - np.min(sig)

    # --- 2. Respiratory Rate (Breaths per minute) ---
    duration_minutes = len(sig) / fs / 60.0
    respiratory_rate = len(peaks) / duration_minutes if duration_minutes > 0 else 0

    # --- 3. Recovery Slope ---
    # Linear fit over the entire signal window to find the general trend/slope
    recovery_slope = np.polyfit(t, sig, 1)[0]

    # --- 7. Skewness of Respiratory Wave ---
    wave_skewness = skew(sig)

    # --- Breath Timing Features ---
    # Calculate Inter-Breath Intervals (IBI)
    if len(peaks) > 1:
        ibi_arrays = np.diff(peaks) / fs
        # 9. Inter-breath interval (Mean)
        inter_breath_interval = np.mean(ibi_arrays)
        # 8. Coefficient of variation of breath duration
        cv_breath_duration = np.std(ibi_arrays) / inter_breath_interval if inter_breath_interval > 0 else 0
    else:
        inter_breath_interval = 0
        cv_breath_duration = 0

    # --- 5 & 6. Inhale/Exhale Ratio and Fractional Inspiratory Time ---
    inhale_durations = []
    exhale_durations = []
    
    # Pair peaks and troughs to find inhale/exhale durations
    for p in peaks:
        # Find the closest trough BEFORE the peak (Inhalation)
        prior_troughs = troughs[troughs < p]
        if len(prior_troughs) > 0:
            inhale_durations.append((p - prior_troughs[-1]) / fs)
            
        # Find the closest trough AFTER the peak (Exhalation)
        next_troughs = troughs[troughs > p]
        if len(next_troughs) > 0:
            exhale_durations.append((next_troughs[0] - p) / fs)

    mean_inhale = np.mean(inhale_durations) if len(inhale_durations) > 0 else 0
    mean_exhale = np.mean(exhale_durations) if len(exhale_durations) > 0 else 0
    total_breath_time = mean_inhale + mean_exhale

    ie_ratio = mean_inhale / mean_exhale if mean_exhale > 0 else 0
    fractional_inspiratory_time = mean_inhale / total_breath_time if total_breath_time > 0 else 0

    # --- Frequency Domain Features ---
    # Compute Power Spectral Density (PSD) using Welch's method
    freqs, psd = welch(sig, fs=fs, nperseg=min(len(sig), 256))
    
    # 11. Power Spectral Density (Peak Power)
    peak_psd = np.max(psd)
    
    # 4. Signal Smoothness (Spectral Entropy)
    # Normalize PSD to create a probability distribution, then calculate Shannon entropy
    psd_norm = psd / np.sum(psd) if np.sum(psd) > 0 else psd
    spectral_entropy = entropy(psd_norm)

    # 10. Dominant Frequency Bandwidth
    # Calculate Full Width at Half Maximum (FWHM) of the dominant frequency peak
    half_max = peak_psd / 2.0
    # Find frequencies where PSD is above half maximum
    above_half = np.where(psd >= half_max)[0]
    if len(above_half) > 1:
        dominant_freq_bw = freqs[above_half[-1]] - freqs[above_half[0]]
    else:
        dominant_freq_bw = 0.0

    # Return exactly the 11 features requested as a numpy array
    return np.array([
        ptp_amplitude, 
        respiratory_rate, 
        recovery_slope, 
        spectral_entropy, 
        ie_ratio, 
        fractional_inspiratory_time, 
        wave_skewness, 
        cv_breath_duration, 
        inter_breath_interval, 
        dominant_freq_bw, 
        peak_psd
    ])

# Test the function on the first valid signal to ensure it compiles and runs
test_person = final_df.index[0]
test_activity = final_df.columns[0]
sample_signal = final_df.loc[test_person, test_activity]

if isinstance(sample_signal, list) and len(sample_signal) > 0:
    test_features = extract_features(sample_signal)
    print(f"Test Extraction Successful for {test_person} ({test_activity})")
    print(f"Feature Vector Shape: {test_features.shape}")
    print(f"Sample Features:\n{np.round(test_features, 4)}")
else:
    print("Test signal was empty. Check dataframe generation.")

PHASE 3: RESPIRATORY FEATURE EXTRACTION
Test Extraction Successful for p1 (rest)
Feature Vector Shape: (11,)
Sample Features:
[ 2.1045e+00  3.0000e+01 -6.0000e-04  4.3558e+00  1.7345e+00  6.3430e-01
  1.9593e+00  2.7430e-01  2.0692e+00  3.6328e+00  3.6370e-01]


In [6]:
print("=" * 60)
print("FEATURE ANALYSIS: MEAN & MEDIAN BY GROUND TRUTH")
print("=" * 60)

import pandas as pd
import numpy as np

# 1. Choose which activity state to analyze (e.g., 'rest', 'walk', 'run', 'stairs')
activity_to_analyze = 'rest' 
print(f"Analyzing physiological features during: '{activity_to_analyze}'\n")

# 2. Extract RAW features (We do not want Z-scores for this table)
features_list = []
valid_pids = []

for pid in final_df.index:
    signal = final_df.loc[pid, activity_to_analyze]
    
    if isinstance(signal, list) and len(signal) > 0:
        feats = extract_features(signal)
        if not np.isnan(feats).any():
            features_list.append(feats)
            valid_pids.append(pid)

feature_names = [
    'ptp_amplitude', 'respiratory_rate', 'recovery_slope', 
    'spectral_entropy', 'ie_ratio', 'fractional_inspiratory_time', 
    'wave_skewness', 'cv_breath_duration', 'inter_breath_interval', 
    'dominant_freq_bw', 'peak_psd'
]

df_raw = pd.DataFrame(features_list, index=valid_pids, columns=feature_names)

# 3. Attach the Ground Truth labels 
# (Re-using the mapping logic from Phase 1/5 to be safe)
def map_label(pid):
    pid_num = int(pid.replace('p', '')) - 1
    return CLASS_NAMES.get(class_map.get(pid_num, 0), 'Normal')

df_raw['Group'] = [map_label(pid) for pid in df_raw.index]

# 4. Filter specifically for Normal, Gym, and Sports (excluding Smokers)
target_groups = ['Normal', 'Gym', 'Sports']
df_filtered = df_raw[df_raw['Group'].isin(target_groups)]

# 5. Calculate Mean and Median using Pandas GroupBy
group_stats = df_filtered.groupby('Group').agg(['mean', 'median'])

# 6. Formatting for readability
# Transpose (.T) flips the table so Features are rows and Groups are columns,
# making it much easier to read inside a Kaggle output cell.
stats_transposed = group_stats.T.round(3)

pd.set_option('display.max_rows', None) # Ensure pandas prints all 11 features
print(stats_transposed)
pd.reset_option('display.max_rows')

# Optional: Quick loop to highlight the biggest differences
print("\n--- Quick Insights ---")
for feature in feature_names:
    means = group_stats[feature]['mean']
    max_group = means.idxmax()
    min_group = means.idxmin()
    print(f"Highest {feature}: {max_group} | Lowest: {min_group}")

FEATURE ANALYSIS: MEAN & MEDIAN BY GROUND TRUTH
Analyzing physiological features during: 'rest'

Group                                  Gym  Normal  Sports
ptp_amplitude               mean     2.316   2.074   2.228
                            median   2.216   1.983   2.187
respiratory_rate            mean    30.510  30.968  29.893
                            median  30.000  30.000  30.000
recovery_slope              mean    -0.000   0.000  -0.000
                            median  -0.000   0.000   0.000
spectral_entropy            mean     4.390   4.406   4.422
                            median   4.399   4.417   4.427
ie_ratio                    mean     1.085   1.195   1.112
                            median   1.048   1.190   1.086
fractional_inspiratory_time mean     0.514   0.533   0.510
                            median   0.512   0.543   0.521
wave_skewness               mean    -0.199   0.031   0.081
                            median  -0.164   0.032  -0.110
cv_breath_duration

In [7]:
print("=" * 60)
print("FEATURE OVERLAP ANALYSIS: TERTILE (1/3) MEANS BY GROUP")
print("=" * 60)

import pandas as pd
import numpy as np

# Define the groups we want to analyze
target_groups = ['Normal', 'Gym', 'Sports']
df_filtered = df_raw[df_raw['Group'].isin(target_groups)]

for group in target_groups:
    print(f"\n{'-'*55}")
    print(f"TERTILE ANALYSIS (INTRA-GROUP SPREAD): {group.upper()}")
    print(f"{'-'*55}")
    
    df_group = df_filtered[df_filtered['Group'] == group]
    tertile_results = {}
    
    for feature in feature_names:
        # Extract the specific feature, drop any NaNs, and sort from lowest to highest
        sorted_vals = df_group[feature].dropna().sort_values().values
        
        # Safety check: if a group is too small, skip it
        if len(sorted_vals) < 3:
            tertile_results[feature] = [np.nan, np.nan, np.nan]
            continue
            
        # Split the sorted data into 3 equal-sized chunks
        chunks = np.array_split(sorted_vals, 3)
        
        # Calculate the mean for each specific chunk
        mean_bottom = chunks[0].mean()
        mean_middle = chunks[1].mean()
        mean_top = chunks[2].mean()
        
        tertile_results[feature] = [mean_bottom, mean_middle, mean_top]
        
    # Convert to a DataFrame for clean formatting
    df_tertiles = pd.DataFrame.from_dict(
        tertile_results, 
        orient='index', 
        columns=['Bottom 1/3 (Mean)', 'Middle 1/3 (Mean)', 'Top 1/3 (Mean)']
    )
    
    # Optional: Sort by the middle third to make the table flow nicely
    df_tertiles = df_tertiles.sort_values(by='Middle 1/3 (Mean)')
    
    print(df_tertiles.round(3))

FEATURE OVERLAP ANALYSIS: TERTILE (1/3) MEANS BY GROUP

-------------------------------------------------------
TERTILE ANALYSIS (INTRA-GROUP SPREAD): NORMAL
-------------------------------------------------------
                             Bottom 1/3 (Mean)  Middle 1/3 (Mean)  \
wave_skewness                           -0.632             -0.010   
recovery_slope                          -0.001              0.000   
cv_breath_duration                       0.169              0.209   
peak_psd                                 0.172              0.318   
fractional_inspiratory_time              0.449              0.544   
ie_ratio                                 0.826              1.198   
ptp_amplitude                            1.549              1.971   
inter_breath_interval                    1.871              1.989   
dominant_freq_bw                         2.741              3.766   
spectral_entropy                         4.356              4.416   
respiratory_rate           

In [9]:
print("=" * 60)
print("PHASE 3.5: BATCH EXTRACTION & NORMALIZATION")
print("=" * 60)

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

def get_normalized_features_for_activity(activity_name):
    """
    Extracts features for all participants for a specific activity 
    and applies Z-score normalization.
    """
    features_list = []
    valid_participants = []

    # 1. Loop through all participants for the specific activity
    for person_id in final_df.index:
        signal = final_df.loc[person_id, activity_name]
        
        # Check if the signal is valid and extracted properly
        if isinstance(signal, list) and len(signal) > 0:
            feats = extract_features(signal)
            
            # Ensure no NaNs were returned (meaning signal was too short)
            if not np.isnan(feats).any():
                features_list.append(feats)
                valid_participants.append(person_id)

    # Convert to a 2D numpy array: Shape (N_participants, 11_features)
    X_raw = np.array(features_list)
    
    # 2. Apply Z-score Normalization (Standardization)
    scaler = StandardScaler()
    X_normalized = scaler.fit_transform(X_raw)
    
    # Create a DataFrame for easy tracking
    feature_names = [
        'ptp_amplitude', 'respiratory_rate', 'recovery_slope', 
        'spectral_entropy', 'ie_ratio', 'fractional_inspiratory_time', 
        'wave_skewness', 'cv_breath_duration', 'inter_breath_interval', 
        'dominant_freq_bw', 'peak_psd'
    ]
    
    df_normalized = pd.DataFrame(X_normalized, index=valid_participants, columns=feature_names)
    
    return df_normalized, scaler

# Example: Extract and normalize features for the 'rest' activity
df_rest_features, rest_scaler = get_normalized_features_for_activity('rest')

print(f"Successfully extracted and normalized features for {len(df_rest_features)} participants.")
print("\nSample of Normalized Features (Mean ~ 0, Std ~ 1):")
print(df_rest_features.head(3).round(3))

PHASE 3.5: BATCH EXTRACTION & NORMALIZATION
Successfully extracted and normalized features for 90 participants.

Sample of Normalized Features (Mean ~ 0, Std ~ 1):
    ptp_amplitude  respiratory_rate  recovery_slope  spectral_entropy  \
p1         -0.126            -0.243          -0.577            -0.881   
p2          1.498            -0.243           1.081            -0.781   
p3         -0.872            -0.243           0.408             1.454   

    ie_ratio  fractional_inspiratory_time  wave_skewness  cv_breath_duration  \
p1     1.739                        1.452          2.637               1.970   
p2     0.554                        0.650         -1.079              -0.107   
p3     0.759                        0.809          1.878               0.949   

    inter_breath_interval  dominant_freq_bw  peak_psd  
p1                  0.434             0.279    -0.130  
p2                  0.625             0.513     0.959  
p3                  0.307            -1.188    -0.495 